In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names
import pandas as pd
import numpy as np

# =====================================================================================
# 📚 데이터셋 정보
# 📚 제목: nayohan/Evol-Instruct-Code-80k-v1-ko
# 📚 의미: 이 데이터셋은 '지시사항(instruction)'과 그에 대한 '최적의 답변(output)' 쌍으로 구성된 대규모 한국어 지식 기반 데이터셋입니다.
# 📚 목적: LLM(대규모 언어 모델)이 특정 질문(instruction)을 받았을 때, 어떻게 논리적이고 구조화된 답변(output)을 생성하는지 학습하는 데 사용됩니다.
# 💡 우리가 할 일: 이 데이터를 분석하여, AI가 얼마나 '똑똑하게' 답변하도록 프롬프트를 설계할 수 있는지 초보자 입장에서 체험해 볼 거예요!
# =====================================================================================

DATASET_NAME = "nayohan/Evol-Instruct-Code-80k-v1-ko"
SAMPLE_COUNT = 10  # 초보자 실습을 위해 상위 10개만 샘플링합니다. (전체 데이터셋을 로드하면 매우 느립니다!)

print("✨✨ 코딩 튜터 AI가 여러분의 AI 코딩 실력을 업그레이드해 줄게요! ✨✨")

# 1. 데이터셋 로딩 (스트리밍 모드 우선 시도)
print("\n--- 🚀 Step 1: 데이터셋 로드 및 준비 (feat. 스트리밍 최적화) ---")
dataset = None
try:
    # 스트리밍 모드는 메모리를 절약하고 빠른 전처리 과정을 돕습니다.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공! 스트리밍 모드(streaming=True)로 데이터셋을 로드했습니다. 메모리 걱정 끝!")
except Exception as e:
    print(f"⚠️ 스트리밍 로드에 실패했습니다 ({e.__class__.__name__}). 일반 로드로 전환합니다.")
    try:
        # 스트리밍이 안 될 경우, 작은 분량의 테스트 데이터만 다운로드하여 진행합니다.
        dataset = load_dataset(DATASET_NAME, split='train')
        print("✅ 성공! 일반 모드(Dataset)로 데이터셋을 로드했습니다.")
    except Exception as e_fallback:
        print(f"❌ 데이터셋 로드에 실패했습니다. 인터넷 연결이나 데이터셋 이름을 확인해주세요: {e_fallback}")
        exit()

# 2. 데이터 샘플링 전략 (스트리밍 vs. 일반 Dataset 처리)
print("\n--- ✨ Step 2: 샘플 데이터 추출 (메모리 효율화) ---")
sample_data_list = []

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    print(f"🏃‍♂️ 스트리밍 모드 감지: .take()를 사용하여 상위 {SAMPLE_COUNT}개의 데이터만 순차적으로 가져옵니다.")
    sample_data_iterator = dataset.take(SAMPLE_COUNT)
    # 메모리에 한 번에 다 로드하기 위해 리스트로 변환합니다.
    sample_data_list = list(sample_data_iterator)
else:
    # 일반 데이터셋 (Dataset)입니다.
    print(f"📦 일반 Dataset 감지: 상위 {SAMPLE_COUNT}개의 데이터 슬라이스를 사용합니다.")
    # Dataset 타입은 select()가 가능하나, 여기서는 list()로 먼저 변환하여 안전하게 처리합니다.
    # Note: Dataset 객체에서 바로 list(dataset.select(...))를 사용합니다.
    sample_data_list = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))


print(f"\n🎉 준비 완료! 총 {len(sample_data_list)}개의 샘플을 분석할 준비가 되었습니다.")


# 3. 초보자를 위한 창의적인 실습: 'AI 답변 구조 분석기' 구현
# 목표: 단순한 텍스트 출력을 넘어, AI가 좋은 답변을 하려면 어떤 구조(단계, 목록, 근거 제시)가 필요한지 코드로 분석해 봅니다.

def analyze_ai_response(sample):
    """
    주어진 Instruction과 Output 쌍을 받아 AI 답변의 구조와 난이도를 분석하는 시뮬레이션 함수.
    """
    instruction = sample['instruction']
    output = sample['output']
    
    # 1. 답변의 길이 (정량적 분석)
    output_length = len(output)
    
    # 2. 답변의 구조 복잡도 판단 (핵심 논리)
    complexity = "단순 답변"
    if "단계별" in instruction or "방법" in instruction:
        complexity = "절차적 설명 (Process)"
    elif "비교" in instruction or "장점과 단점" in instruction:
        complexity = "분석적 비교 (Compare & Contrast)"
    elif "요약" in instruction:
        complexity = "정보 추출/요약 (Extraction)"
    else:
        complexity = "일반 Q&A"
        
    # 3. 답변의 형식성 판단 (Structure Check)
    # 답변에 번호 목록이나 명확한 구분이 있는지 확인합니다.
    is_structured = (
        ("번호" in output) or 
        ("list" in output.lower()) or 
        (len(output) > 50 and ("\n" in output))
    )
    
    return {
        "Task Type": complexity,
        "Is Structured": "✅ 예" if is_structured else "❌ 아니요",
        "Output Length (Char)": output_length
    }

print("\n=============================================================================")
print("✨ 🧪 실습 시작: AI 답변 구조 분석기 (Prompt Pattern Detector) 🧪 ✨")
print("=============================================================================")

# 4. 실습 실행 및 결과 출력
print("\n💡 원리 이해하기:")
print("AI가 좋은 답변을 하려면, 단순히 답만 주는 것이 아니라 '어떻게' 구조화해서 설명하는지가 중요합니다. 이 코드는 그 구조적 특징을 분석합니다.")

results = []
for i, sample in enumerate(sample_data_list):
    analysis = analyze_ai_response(sample)
    results.append(analysis)
    
    # 터미널 출력을 보기 좋게 만듭니다.
    print(f"\n--- [{i+1}/{len(sample_data_list)}] ✨ 분석 대상 샘플 ✨ ---")
    print(f"  [❓ 명령어(Instruction)]: {sample['instruction'][:40]}...")
    print(f"  [🧠 결과(Output)]: {sample['output'][:50]}...")
    
    print("  --- [분석 결과] ---")
    for key, value in analysis.items():
        print(f"  - {key:<20}: {value}")

print("\n=============================================================================")
print("🎉🎊 실습 완료! 튜터의 한마디 🎊🎉")
print("-----------------------------------------------------------------------------")
print("축하합니다! 여러분은 이제 데이터 속에서 AI의 '패턴'을 읽어내는 분석 능력을 키웠어요.")
print("✅ 오늘 배운 것:")
print("1. 스트리밍 방식으로 큰 데이터셋을 효율적으로 로드하는 방법.")
print("2. 단순 텍스트가 아닌, 답변의 '구조(Structure)'와 '복잡도(Complexity)'를 분석하는 시야.")
print("\n✨ 다음에 더 나아가면, 이 분석 결과를 바탕으로 '이런 질문을 받았을 때 이렇게 답해줘!'라고 AI에게 명령하는 고급 프롬프트 엔지니어링을 할 수 있답니다!")
print("-----------------------------------------------------------------------------")